In [1]:
from langchain_ollama import ChatOllama

f:\SAMARPAN\GenAI projects\Natural Language to SQL\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [46]:
model1 = ChatOllama(
    model="hf.co/unsloth/Qwen3-Coder-30B-A3B-Instruct-GGUF:UD-Q4_K_XL",
    temperature=0
)

In [47]:
print(model1.invoke("tell me a joke"))

content="Why don't scientists trust atoms?\n\nBecause they make up everything!" additional_kwargs={} response_metadata={'model': 'hf.co/unsloth/Qwen3-Coder-30B-A3B-Instruct-GGUF:UD-Q4_K_XL', 'created_at': '2026-04-28T16:33:29.1841305Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2038262900, 'load_duration': 117405700, 'prompt_eval_count': 12, 'prompt_eval_duration': 545794400, 'eval_count': 14, 'eval_duration': 1347327200, 'logprobs': None, 'model_name': 'hf.co/unsloth/Qwen3-Coder-30B-A3B-Instruct-GGUF:UD-Q4_K_XL', 'model_provider': 'ollama'} id='lc_run--019dd4f0-3c64-7cd2-83dc-ebe430946e46-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 12, 'output_tokens': 14, 'total_tokens': 26}


In [34]:
from langchain_community.utilities.sql_database import SQLDatabase
from langchain_community.tools import QuerySQLDataBaseTool
from langchain_classic.chains import create_sql_query_chain
import os
from dotenv import load_dotenv
load_dotenv()

True

In [35]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen3-Coder-30B-A3B-Instruct",
    task="text-generation",
    huggingfacehub_api_token= os.getenv("HUGGINGFACEHUB_ACCESS_TOKEN")
)

model2= ChatHuggingFace(llm=llm)

In [36]:
db_user = os.getenv("db_user")
db_password = os.getenv("db_password")
db_host = os.getenv("db_host")
db_name = os.getenv("db_name")
db_port = os.getenv("db_port")

connection_string = f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"

db= SQLDatabase.from_uri(connection_string)

In [76]:
question = "Most profitable order along with the profits gained"

In [77]:
query_generator= create_sql_query_chain(model1,db) 
print(query_generator.invoke({"question": question}))

Question: Most profitable order along with the profits gained
SQLQuery: SELECT 
    o.ordernumber,
    SUM((od.priceeach - p.buyprice) * od.quantityordered) AS profit
FROM orders o
JOIN orderdetails od ON o.ordernumber = od.ordernumber
JOIN products p ON od.productcode = p.productcode
GROUP BY o.ordernumber
ORDER BY profit DESC
LIMIT 1


In [78]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from pydantic_models.query import Query

parser = PydanticOutputParser(pydantic_object=Query)
extraction_template=PromptTemplate(
    template=" extract only the sql query from the given text: \n {text} \n{format_instruction}",
    input_variables=["text"],
    partial_variables={"format_instruction":parser.get_format_instructions()}
)

In [79]:
from langchain_classic.schema.runnable import RunnableLambda

simple_chain = query_generator | extraction_template | model1 | parser | RunnableLambda(lambda x : x.sql)
query=simple_chain.invoke({"question":question})

In [80]:
query_executor = QuerySQLDataBaseTool(db=db)

sql_result = query_executor.invoke(query)
print(sql_result)

[(10165, Decimal('26465.57'))]


In [81]:
from langchain_core.prompts import PromptTemplate

model3=ChatOllama(
    model="gpt-oss:latest"
)

rephraser_template=PromptTemplate(
    template="""
            Given the following user question, corresponding sql query and sql result, answer the user question:
            Question:{question}
            SQL Query:{query}
            SQL Result:{result}
            Answer:
""",
input_variables=['question', 'query', 'result']
)

rephraser_chain = rephraser_template | model3 | StrOutputParser()

print(rephraser_chain.invoke({
    "question" : question,
    "query" : query,
    "result" : sql_result
}))

**Answer**

The single most profitable order is:

| Order Number | Profit |
|--------------|--------|
| **10165** | **$26,465.57** |

The profit was calculated as the sum over all products in that order of the difference between the selling price (`priceeach`) and the purchase cost (`buyprice`), multiplied by the quantity ordered. This yields the total profit of **$26,465.57** for order **10165**.
